# 02 — K-means

Agrège les CSV par `SubjectID`, prépare les variables numériques, puis lance un K-means.

Les fichiers utiles sont sauvegardés dans `analysis/`.

## 1. Configuration

In [ ]:
from pathlib import Path
from functools import reduce
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def detect_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Chaine",
        Path.cwd() / "chaine_extracted" / "Chaine",
        Path.cwd().parent / "Chaine",
    ]
    for candidate in candidates:
        if (candidate / "csv").exists() and (
            (candidate / "data" / "MainTable.csv").exists() or (candidate / "scripts").exists()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        "Impossible de détecter automatiquement le dossier Chaine. "
        "Modifiez PROJECT_DIR manuellement dans cette cellule."
    )

# Option manuelle si besoin.
# PROJECT_DIR = Path(r"C:/Users/.../Chaine")
PROJECT_DIR = detect_project_dir()

CSV_DIR = PROJECT_DIR / "csv"
ANALYSIS_DIR = PROJECT_DIR / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# Agrégation des métriques : "outer" garde tous les étudiants, "inner" garde l’intersection.
JOIN_MODE = "outer"  # "outer" ou "inner"

# Gestion des valeurs manquantes : "median", "mean" ou "drop_rows".
IMPUTATION = "median"

# Minimum de variables observées par étudiant.
MIN_NON_NULL_FEATURES = 2

# Variables à exclure si besoin.
DROP_FEATURES = []

# Paramètres.
N_CLUSTERS = 4
K_RANGE = range(2, 9)  # valeurs testées pour le graphique coude/silhouette
# Reproductibilité.
RANDOM_STATE = 42
N_INIT = 20

# Variables affichées dans le profil.
TOP_N_PROFILE_VARIABLES = 12

print("Projet :", PROJECT_DIR)
print("Dossier CSV :", CSV_DIR)
print("Dossier analyse :", ANALYSIS_DIR)
print("Nombre de clusters choisi :", N_CLUSTERS)


## 2. Chargement des CSV

In [ ]:
if not CSV_DIR.exists():
    raise FileNotFoundError(f"Dossier CSV introuvable : {CSV_DIR}")

csv_paths = sorted(CSV_DIR.glob("*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"Aucun fichier CSV trouvé dans : {CSV_DIR}")

tables = []
inventory = []

for path in csv_paths:
    # CSV à ignorer.
    if path.name.lower() in {"run_report.csv", "stats.csv"}:
        inventory.append({
            "fichier": path.name,
            "statut": "ignoré",
            "raison": "fichier de rapport/statistiques, pas une métrique SubjectID",
            "lignes": None,
            "colonnes": None,
        })
        continue

    try:
        df = pd.read_csv(path)
    except Exception as exc:
        inventory.append({
            "fichier": path.name,
            "statut": "ignoré",
            "raison": f"lecture impossible : {exc}",
            "lignes": None,
            "colonnes": None,
        })
        continue

    if "SubjectID" not in df.columns:
        inventory.append({
            "fichier": path.name,
            "statut": "ignoré",
            "raison": "pas de colonne SubjectID",
            "lignes": df.shape[0],
            "colonnes": df.shape[1],
        })
        continue

    # Un étudiant par ligne.
    df = df.drop_duplicates(subset=["SubjectID"], keep="first")

    # Renommage en cas de collision.
    feature_cols = [c for c in df.columns if c != "SubjectID"]
    renamed = {}
    for c in feature_cols:
        if c in {"rows", "columns", "status"}:
            renamed[c] = f"{path.stem}_{c}"
    if renamed:
        df = df.rename(columns=renamed)

    tables.append(df)

    inventory.append({
        "fichier": path.name,
        "statut": "chargé",
        "raison": "",
        "lignes": df.shape[0],
        "colonnes": df.shape[1],
    })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)

if not tables:
    raise ValueError("Aucun CSV de métrique exploitable n'a été trouvé.")

features = reduce(lambda left, right: pd.merge(left, right, on="SubjectID", how=JOIN_MODE), tables)
print(f"Table agrégée : {features.shape[0]:,} sujets × {features.shape[1]-1:,} variables")
display(features.head())


## 3. Nettoyage

In [ ]:
features = features.copy()

# Conversion numérique.
for col in features.columns:
    if col != "SubjectID":
        features[col] = pd.to_numeric(features[col], errors="coerce")

# Exclusions.
drop_manual = [c for c in DROP_FEATURES if c in features.columns]
if drop_manual:
    features = features.drop(columns=drop_manual)

numeric_cols = [
    c for c in features.columns
    if c != "SubjectID" and pd.api.types.is_numeric_dtype(features[c])
]

X_raw = features[["SubjectID", *numeric_cols]].copy()

# Garde les lignes assez complètes.
non_null_counts = X_raw[numeric_cols].notna().sum(axis=1)
X_raw = X_raw.loc[non_null_counts >= MIN_NON_NULL_FEATURES].reset_index(drop=True)

# Enlève les variables inutilisables.
usable_cols = []
removed_cols = []
for col in numeric_cols:
    series = X_raw[col]
    if series.notna().sum() == 0:
        removed_cols.append((col, "entièrement vide"))
    elif series.nunique(dropna=True) <= 1:
        removed_cols.append((col, "constante"))
    else:
        usable_cols.append(col)

X_raw = X_raw[["SubjectID", *usable_cols]].copy()

print(f"Sujets conservés : {X_raw.shape[0]:,}")
print(f"Variables conservées : {len(usable_cols):,}")
if removed_cols:
    display(pd.DataFrame(removed_cols, columns=["variable", "raison_suppression"]))

missing_summary = (
    X_raw[usable_cols]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("pct_valeurs_manquantes")
    .reset_index()
    .rename(columns={"index": "variable"})
)
display(missing_summary)


## 4. Préparation des données

In [ ]:
if len(usable_cols) < 2:
    raise ValueError("Il faut au moins deux variables numériques non constantes pour réaliser un K-means.")

X_values = X_raw[usable_cols].copy()

if IMPUTATION == "drop_rows":
    keep_mask = X_values.notna().all(axis=1)
    X_raw_kmeans = X_raw.loc[keep_mask].reset_index(drop=True)
    X_values = X_raw_kmeans[usable_cols].copy()
else:
    strategy = {"median": "median", "mean": "mean"}.get(IMPUTATION)
    if strategy is None:
        raise ValueError("IMPUTATION doit valoir 'median', 'mean' ou 'drop_rows'.")
    imputer = SimpleImputer(strategy=strategy)
    X_imputed = imputer.fit_transform(X_values)
    X_values = pd.DataFrame(X_imputed, columns=usable_cols)
    X_raw_kmeans = X_raw.copy()

if X_values.shape[0] < 2:
    raise ValueError("Il faut au moins deux sujets pour lancer un K-means.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

features_for_kmeans = pd.concat(
    [X_raw_kmeans[["SubjectID"]].reset_index(drop=True), X_values.reset_index(drop=True)],
    axis=1
)
features_for_kmeans.to_csv(ANALYSIS_DIR / "features_for_kmeans.csv", index=False)

print(f"Table utilisée pour K-means : {features_for_kmeans.shape[0]:,} sujets × {len(usable_cols):,} variables")
display(features_for_kmeans.head())


## 5. Choix de k

In [ ]:
n_samples = X_scaled.shape[0]
valid_k_values = [k for k in K_RANGE if 2 <= k <= n_samples - 1]

if not valid_k_values:
    print("Pas assez de sujets pour calculer une silhouette sur plusieurs valeurs de k.")
    model_selection_df = pd.DataFrame(columns=["k", "inertia", "silhouette"])
else:
    rows = []
    for k in valid_k_values:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
        labels = km.fit_predict(X_scaled)
        rows.append({
            "k": k,
            "inertia": km.inertia_,
            "silhouette": silhouette_score(X_scaled, labels),
        })

    model_selection_df = pd.DataFrame(rows)
    display(model_selection_df)

model_selection_df.to_csv(ANALYSIS_DIR / "kmeans_model_selection.csv", index=False)


In [ ]:
if not model_selection_df.empty:
    plt.figure(figsize=(7, 4))
    plt.plot(model_selection_df["k"], model_selection_df["inertia"], marker="o")
    plt.xlabel("Nombre de clusters k")
    plt.ylabel("Inertie")
    plt.title("K-means — méthode du coude")
    plt.xticks(model_selection_df["k"])
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(model_selection_df["k"], model_selection_df["silhouette"], marker="o")
    plt.xlabel("Nombre de clusters k")
    plt.ylabel("Score de silhouette")
    plt.title("K-means — score de silhouette")
    plt.xticks(model_selection_df["k"])
    plt.tight_layout()
    plt.show()
else:
    print("Graphiques de sélection de k non disponibles.")


## 6. K-means final

In [ ]:
if not (2 <= N_CLUSTERS <= X_scaled.shape[0]):
    raise ValueError(f"N_CLUSTERS doit être compris entre 2 et le nombre de sujets ({X_scaled.shape[0]}).")

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=N_INIT)
labels = kmeans.fit_predict(X_scaled)
cluster_labels = labels + 1  # numérotation plus lisible : 1, 2, 3, ...

clusters_df = pd.DataFrame({
    "SubjectID": X_raw_kmeans["SubjectID"].values,
    "Cluster": cluster_labels,
})

features_with_clusters = features_for_kmeans.copy()
features_with_clusters.insert(1, "Cluster", cluster_labels)

centers_scaled_df = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=usable_cols,
    index=[f"Cluster {i}" for i in range(1, N_CLUSTERS + 1)]
)
centers_scaled_df.index.name = "Cluster"

centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
centers_original_df = pd.DataFrame(
    centers_original,
    columns=usable_cols,
    index=[f"Cluster {i}" for i in range(1, N_CLUSTERS + 1)]
)
centers_original_df.index.name = "Cluster"

cluster_summary_df = (
    clusters_df["Cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .reset_index(name="n_sujets")
)
cluster_summary_df["pct_sujets"] = cluster_summary_df["n_sujets"] / cluster_summary_df["n_sujets"].sum() * 100

clusters_df.to_csv(ANALYSIS_DIR / "kmeans_clusters.csv", index=False)
features_with_clusters.to_csv(ANALYSIS_DIR / "features_with_kmeans_clusters.csv", index=False)
centers_scaled_df.to_csv(ANALYSIS_DIR / "kmeans_cluster_centers_scaled.csv")
centers_original_df.to_csv(ANALYSIS_DIR / "kmeans_cluster_centers_original_units.csv")
cluster_summary_df.to_csv(ANALYSIS_DIR / "kmeans_cluster_summary.csv", index=False)

print("Répartition des sujets par cluster")
display(cluster_summary_df)

print("Centres des clusters dans les unités originales")
display(centers_original_df)

print("Fichiers sauvegardés dans :", ANALYSIS_DIR)


## 7. Taille des clusters

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(cluster_summary_df["Cluster"].astype(str), cluster_summary_df["n_sujets"])
plt.xlabel("Cluster")
plt.ylabel("Nombre de sujets")
plt.title("K-means — taille des clusters")
plt.tight_layout()
plt.show()


## 8. Profil des clusters

In [ ]:
profile_rows = []
for cluster_name, row in centers_scaled_df.iterrows():
    tmp = row.abs().sort_values(ascending=False).head(TOP_N_PROFILE_VARIABLES)
    for variable in tmp.index:
        profile_rows.append({
            "Cluster": cluster_name,
            "variable": variable,
            "centre_standardise": row[variable],
            "sens": "au-dessus de la moyenne" if row[variable] > 0 else "en dessous de la moyenne",
        })

cluster_profile_df = pd.DataFrame(profile_rows)
cluster_profile_df.to_csv(ANALYSIS_DIR / "kmeans_cluster_profile.csv", index=False)
display(cluster_profile_df)


In [ ]:
# Variables les plus discriminantes.
if len(usable_cols) > 0:
    cols_to_plot = (
        centers_scaled_df
        .abs()
        .max(axis=0)
        .sort_values(ascending=False)
        .head(min(TOP_N_PROFILE_VARIABLES, len(usable_cols)))
        .index
        .tolist()
    )

    heatmap_data = centers_scaled_df[cols_to_plot]

    plt.figure(figsize=(max(8, 0.65 * len(cols_to_plot)), 1.2 + 0.5 * N_CLUSTERS))
    im = plt.imshow(heatmap_data, aspect="auto", vmin=-2, vmax=2)
    plt.colorbar(im, fraction=0.046, pad=0.04, label="Centre standardisé")
    plt.xticks(range(len(cols_to_plot)), cols_to_plot, rotation=90)
    plt.yticks(range(N_CLUSTERS), heatmap_data.index)
    plt.title("K-means — profil des clusters")
    plt.tight_layout()
    plt.show()


## 9. Résumé

In [ ]:
def format_cluster_profile(cluster_name, n=5):
    tmp = centers_scaled_df.loc[cluster_name].copy()
    tmp = tmp.reindex(tmp.abs().sort_values(ascending=False).head(n).index)
    parts = []
    for variable, value in tmp.items():
        sign = "+" if value >= 0 else "-"
        parts.append(f"{variable} ({sign}{abs(value):.2f})")
    return ", ".join(parts)

print("Résumé K-means")
print("--------------")
print(f"Nombre de sujets inclus : {clusters_df.shape[0]}")
print(f"Nombre de variables incluses : {len(usable_cols)}")
print(f"Nombre de clusters choisi : {N_CLUSTERS}")
print()

for _, row in cluster_summary_df.iterrows():
    cluster_name = f"Cluster {int(row['Cluster'])}"
    print(f"{cluster_name} : {int(row['n_sujets'])} sujets ({row['pct_sujets']:.1f} %)")
    print("  Variables principales : " + format_cluster_profile(cluster_name))
    print()
